# Vancouver Crime Type Classifier

## Overview
This project applies K-Nearest Neighbours (KNN) classification to predict 
crime type based on historical incident data from the Vancouver Police 
Department. Model performance is evaluated on a held-out test set to ensure 
generalisation to unseen data.

## Data Source
- **Provider:** Vancouver Police Department (VPD) GeoDASH Open Data
- **Dataset:** Crime data from 2003 to present, updated weekly
- **URL:** https://geodash.vpd.ca/opendata/

## Features Used
| Feature | Description |
|---|---|
| `NEIGHBOURHOOD` | Vancouver neighbourhood where incident occurred |
| `HOUR` | Hour of day (0-23) |
| `MONTH` | Month of year (1-12) |
| `DAY_OF_WEEK` | Day of week derived from date |
| `IS_WEEKEND` | Whether incident occurred on a weekend |
| `SEASON` | Season derived from month |
| `YEAR` | Year of incident |

## Target Variable
`TYPE` — Crime category (e.g. Theft, Break & Enter, Assault, Mischief)

## Methodology
1. Data loading and exploration
2. Data cleaning and feature engineering
3. Train/test split
4. KNN model training
5. Model evaluation and results

## Tools & Libraries
- **Language:** R
- **Environment:** JupyterLab
- **Key packages:** tidyverse, class, caret

In [2]:
install.packages(c(
  "tidyverse",   
  "caret",   
  "class",       
  "lubridate"    
))


The downloaded binary packages are in
	/var/folders/xy/cz8fxbvn7mv8c_d1fxb7sl5r0000gn/T//RtmpE8Jbep/downloaded_packages


In [3]:
library(tidyverse)
library(caret)
library(class)
library(lubridate)

Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘lubridate’ was built under R version 4.4.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.2     ✔ tibble    3.2.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: lattice


Attaching package: ‘caret’


The following object is masked from ‘package:purrr’:

    lift




In [6]:
crimes <- read_csv("/Users/kathyzhao/Crime-Classifier/crimedata_csv_AllNeighbourhoods_AllYears.csv")

Rows: 913777 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): TYPE, HUNDRED_BLOCK, NEIGHBOURHOOD
dbl (7): YEAR, MONTH, DAY, HOUR, MINUTE, X, Y

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [15]:
head(crimes)

TYPE,YEAR,MONTH,DAY,HOUR,MINUTE,HUNDRED_BLOCK,NEIGHBOURHOOD,X,Y,DATE,DAY_OF_WEEK
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<date>,<chr>
Break and Enter Commercial,2023,9,14,3,30,10XX ALBERNI ST,West End,491065.3,5459130,2023-09-14,Thursday
Break and Enter Commercial,2024,2,24,4,8,10XX BARCLAY ST,West End,490865.2,5458841,2024-02-24,Saturday
Break and Enter Commercial,2023,4,1,4,7,10XX BEACH AVE,West End,490197.9,5458239,2023-04-01,Saturday
Break and Enter Commercial,2025,4,28,12,5,10XX BEACH AVE,West End,490227.2,5458210,2025-04-28,Monday
Break and Enter Commercial,2023,5,11,18,0,10XX BEACH AVE,Central Business District,490249.2,5458167,2023-05-11,Thursday
Break and Enter Commercial,2024,2,11,22,5,10XX BEACH AVE,Central Business District,490249.2,5458167,2024-02-11,Sunday


## Data Preparation

The raw dataset contains separate YEAR, MONTH, and DAY columns. These are 
combined into a full date to extract DAY_OF_WEEK, since the same day number 
falls on different weekdays across years. DAY_OF_WEEK is a stronger predictor 
of crime type than the raw day number alone.

In [16]:
crimes <- crimes |> 
    mutate(DATE = as_date(paste(YEAR, MONTH, DAY, sep = "-")),
          DAY_OF_WEEK = weekdays(DATE))
head(crimes)

TYPE,YEAR,MONTH,DAY,HOUR,MINUTE,HUNDRED_BLOCK,NEIGHBOURHOOD,X,Y,DATE,DAY_OF_WEEK
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<date>,<chr>
Break and Enter Commercial,2023,9,14,3,30,10XX ALBERNI ST,West End,491065.3,5459130,2023-09-14,Thursday
Break and Enter Commercial,2024,2,24,4,8,10XX BARCLAY ST,West End,490865.2,5458841,2024-02-24,Saturday
Break and Enter Commercial,2023,4,1,4,7,10XX BEACH AVE,West End,490197.9,5458239,2023-04-01,Saturday
Break and Enter Commercial,2025,4,28,12,5,10XX BEACH AVE,West End,490227.2,5458210,2025-04-28,Monday
Break and Enter Commercial,2023,5,11,18,0,10XX BEACH AVE,Central Business District,490249.2,5458167,2023-05-11,Thursday
Break and Enter Commercial,2024,2,11,22,5,10XX BEACH AVE,Central Business District,490249.2,5458167,2024-02-11,Sunday
